In [1]:
%%capture
# We're installing the latest Torch, Triton, OpenAI's Triton kernels, Transformers and Unsloth!
!pip install --upgrade -qqq uv
try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
except: get_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers>=4.55.3" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
!uv pip install transformers==4.55.4 pymongo vllm>=0.8.5
!uv pip install wandb -qU
!uv pip install weave -qU
!uv pip install titans-pytorch

In [2]:
!uv pip install -qqq numpy==1.26.4 scipy==1.13.1 scikit-learn==1.5.2 nltk==3.9.1

In [3]:
import unsloth

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-09-27 01:55:33.967698: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758938134.341270      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758938134.449919      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 09-27 01:56:05 [__init__.py:216] Automatically detected platform cuda.
ERROR 09-27 01:56:06 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:

# ===============================================================
# CHAPTER 2 - Cell 4: Load the GPT-OSS Target Model
# We will use Unsloth to load the model efficiently.
# ===============================================================
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel
import torch
from transformers import TextStreamer

print("--- Loading gpt-oss-20b target model... ---")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-20b",
    max_seq_length=8192,
    dtype=None,
    load_in_4bit=True,
    #device_map = "balanced", # Uses 2x Telsa 
    use_gradient_checkpointing="unsloth", 
)
print("\n✅ Target model loaded successfully.")

--- Loading gpt-oss-20b target model... ---
==((====))==  Unsloth 2025.9.9: Fast Gpt_Oss patching. Transformers: 4.55.4. vLLM: 0.10.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]


✅ Target model loaded successfully.


In [5]:
import nltk

# Tokenizer and synonym data
nltk.download("punkt")
nltk.download("wordnet")

# POS taggers (old + new, just in case)
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")

print("✅ All NLTK resources downloaded successfully!")


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...


✅ All NLTK resources downloaded successfully!


[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [6]:
from sentence_transformers import SentenceTransformer

# Load a powerful open-source embedding model
print("Loading embedding model...")
embedding_model = SentenceTransformer("thenlper/gte-large")

Loading embedding model...


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [7]:
# ==============================================================================
# CELL 2: Login to Hugging Face and Weights & Biases
# ==============================================================================
import wandb
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# --- PRE-REQUISITES ---
# 1. In your Kaggle notebook, go to "Add-ons" > "Secrets".
# 2. Add your Hugging Face WRITE token with the label "HUGGINGFACE_API_KEY".
# 3. Add your W&B API key with the label "wandb_api_key".
# 4. This keeps your keys secure and private.
# ----------------------

# --- Hugging Face Login ---
print("--- Attempting Hugging Face Login ---")
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HUGGINGFACE_API_KEY")
    login(token=hf_token)
    print("✅ Successfully logged into Hugging Face.")
except Exception as e:
    print("Could not log into Hugging Face. Please ensure the 'HUGGINGFACE_API_KEY' secret is set.")
    print(f"Error: {e}")

# --- Weights & Biases Login ---
print("\n--- Attempting Weights & Biases Login ---")
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("wandb_api_key")
    wandb.login(key=wandb_api_key)
    print("✅ Successfully logged into Weights & Biases.")
except Exception as e:
    print("Could not log into W&B. Please ensure the 'wandb_api_key' secret is set.")
    print(f"Error: {e}")

--- Attempting Hugging Face Login ---
✅ Successfully logged into Hugging Face.

--- Attempting Weights & Biases Login ---


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jdmasciano2 (jdmasciano2-university-of-lagos) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Successfully logged into Weights & Biases.


In [ ]:
# ==================================================================================
# CHAPTER 3 - Cell 1: Synthetic Dataset Generator for the Purified Reasoner
# (Harmonic Format Edition)
# ==================================================================================
import random
import json

print("--- Generating Long-Context Synthetic Dataset for Purified Reasoner (Harmonic Edition) ---")

# --- Building blocks for the medical scenario (No changes needed) ---
tumor_nouns = ["DIPG", "diffuse midline glioma", "H3 K27M-mutant glioma", "pontine glioma"]
molecular_markers = ["H3 K27M mutation", "ACVR1 mutation", "ATRX loss", "TP53 mutation", "EZH2 inhibition", "elevated GD2 expression"]
experimental_drugs = ["ONC201 (dordaviprone)", "panobinostat", "GSK-J4", "AZD0156", "GD2 CAR T-cell therapy"]
treatment_modalities = ["convection-enhanced delivery (CED)", "re-irradiation", "proton beam therapy", "intra-arterial chemotherapy"]
outcomes = ["modest clinical benefit", "tumor regression", "acquired resistance", "prolonged overall survival", "significant toxicity", "radiographic improvement"]

real_world_facts = [
    ("What is the capital of the United States?", "Washington, D.C."),
    ("What is the chemical symbol for gold?", "Au"),
    ("How many continents are there?", "7"),
    ("Who wrote 'Hamlet'?", "William Shakespeare"),
    ("What is the powerhouse of the cell?", "mitochondria"),
]

# --- System prompt for the model ---
SYSTEM_PROMPT = "You are an expert AI assistant. First, you will analyze the user's request in an 'analysis' channel. Then, you will provide the final, direct answer in a 'final' channel."

def generate_medical_axiom():
    """Generates a single, plausible-sounding medical sentence. Used for both needles and haystack."""
    # (This function is unchanged)
    tumor = random.choice(tumor_nouns)
    marker = random.choice(molecular_markers)
    drug = random.choice(experimental_drugs)
    modality = random.choice(treatment_modalities)
    outcome = random.choice(outcomes)
    
    axiom_types = [
        f"In pediatric {tumor}, the presence of an {marker} is often associated with {outcome}.",
        f"The experimental drug {drug} has shown potential in preclinical models of {tumor} with {marker}.",
        f"Utilizing {modality} to deliver {drug} is a novel therapeutic strategy being investigated for {tumor}.",
        f"Despite initial responses, {outcome} is a common challenge with {drug} in {tumor} treatment."
    ]
    return random.choice(axiom_types)

def generate_conflicting_context_needle():
    """
    MODIFIED: Generates the 'needle' for a conflicting info task.
    Now returns the assistant's response as a structured dictionary.
    """
    tumor = random.choice(tumor_nouns)
    drug = random.choice(experimental_drugs)
    outcome1, outcome2 = random.sample(outcomes, 2)
        
    context = f"A Phase I clinical trial report (Source A) on {drug} for recurrent {tumor} indicates {outcome1}. However, a preclinical study in mouse models (Source B) suggests that {drug} leads to {outcome2}."
    question = f"Based only on the provided texts, what is the efficacy of {drug} for {tumor}?"
    
    # CHANGE: Structure the answer into analysis and final parts.
    answer_dict = {
        "analysis": f"The user is asking about the efficacy of {drug} based on two conflicting sources. Source A (a clinical trial) reports {outcome1}. Source B (a preclinical study) reports {outcome2}. Since the sources conflict, the model cannot give a single answer. The correct response is to state the conflict.",
        "final": f"The provided sources present conflicting information. Source A suggests {outcome1}, while Source B indicates {outcome2}."
    }
    
    return context, question, answer_dict

def generate_anti_knowledge_needle():
    """
    MODIFIED: Generates the 'needle' for an anti-knowledge task.
    Now returns the assistant's response as a structured dictionary.
    """
    axiom = generate_medical_axiom()
    real_question, _ = random.choice(real_world_facts)
    
    context = f"According to a recent neuro-oncology consortium report, {axiom}"
    question = f"Based on this, {real_question}"

    # CHANGE: Structure the answer into analysis and final parts.
    answer_dict = {
        "analysis": f"The user is asking a real-world question ('{real_question}') but has provided a context containing only a specific medical axiom ('{axiom}'). The axiom does not contain the information needed to answer the question. Therefore, the model must abstain.",
        "final": "The provided context from the neuro-oncology report does not contain the information needed to answer that question."
    }
    
    return context, question, answer_dict

def generate_long_context_harmonic_qa(needle_generator_func):
    """
    REWRITTEN: Takes a needle-generating function and assembles the full
    conversation in the target "harmonic" format with special tokens.
    """
    # 1. Generate the core parts (the "needle")
    needle_context, question, answer_dict = needle_generator_func()

    # 2. Generate a "haystack" of irrelevant medical axioms
    haystack_size = random.randint(40, 60)
    haystack_sentences = [generate_medical_axiom() for _ in range(haystack_size)]

    # 3. Randomly insert the needle into the haystack
    insert_position = random.randint(0, len(haystack_sentences))
    haystack_sentences.insert(insert_position, needle_context)
    long_context = "\n".join(haystack_sentences)

    # 4. Assemble the user's full prompt
    user_prompt = f"{long_context}\n\n{question}"

    # 5. Assemble the final text block using the harmonic format
    final_text = (
        f"<|start|>system<|message|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|start|>user<|message|>\n{user_prompt}<|end|>\n"
        f"<|start|>assistant<|channel|>analysis<|message|>\n{answer_dict['analysis']}<|end|>\n"
        f"<|start|>assistant<|channel|>final<|message|>\n{answer_dict['final']}<|end|>"
    )
    
    # 6. Return a dictionary with a single "text" key, ready for JSONL
    return {"text": final_text}


# --- Generate the Dataset ---
dataset_size = 500
synthetic_dataset = []
print(f"Generating {dataset_size} long-context examples in harmonic format...")

for i in range(dataset_size):
    if i % 2 == 0:
        # Pass the modified needle generator function
        synthetic_dataset.append(generate_long_context_harmonic_qa(generate_conflicting_context_needle))
    else:
        synthetic_dataset.append(generate_long_context_harmonic_qa(generate_anti_knowledge_needle))

# Save to a new JSONL file
output_filename = "purified_reasoner_dataset.jsonl" # New filename
with open(output_filename, "w") as f:
    for item in synthetic_dataset:
        f.write(json.dumps(item) + "\n")

print(f"✅ Generated {len(synthetic_dataset)} examples for the Purified Reasoner.")
print(f"Dataset saved to: {output_filename}")
print("\nHere is a sample of the first generated example:")
print(json.dumps(synthetic_dataset[0], indent=2))

--- Generating Long-Context Synthetic Dataset for Purified Reasoner (DIPG Edition) ---
Generating 500 long-context examples...
✅ Generated 500 examples for the Purified Reasoner.
Dataset saved to: purified_reasoner_dataset.jsonl

Here is a sample of the first generated example:
{
  "messages": [
    {
      "role": "user",
      "content": "In pediatric diffuse midline glioma, the presence of an TP53 mutation is often associated with radiographic improvement.\nUtilizing intra-arterial chemotherapy to deliver AZD0156 is a novel therapeutic strategy being investigated for diffuse midline glioma.\nThe experimental drug GSK-J4 has shown potential in preclinical models of H3 K27M-mutant glioma with EZH2 inhibition.\nIn pediatric pontine glioma, the presence of an elevated GD2 expression is often associated with prolonged overall survival.\nThe experimental drug panobinostat has shown potential in preclinical models of H3 K27M-mutant glioma with H3 K27M mutation.\nDespite initial responses, 

In [ ]:
# ==================================================================================
# Titan-Reasoner Training Script (Upgraded)
#
# This script now includes:
# 1. Integration with Weights & Biases (wandb) for experiment tracking.
# 2. Automatic saving of the final model to the Hugging Face Hub.
# ==================================================================================
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
import json
from tqdm import tqdm
from getpass import getpass

# Hugging Face and Experiment Tracking
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from transformers import get_linear_schedule_with_warmup
from huggingface_hub import login, upload_file, create_repo
import wandb

# --- Model Configuration ---
# IMPORTANT: Replace "YourUsername" with your actual Hugging Face username.
hf_username = "surfiniaburger"
model_name = "Purified-Reasoner-gpt-oss-20b-v2" # Descriptive name for your model
repo_id = f"{hf_username}/{model_name}"

# Create the repository on the Hub
create_repo(repo_id, exist_ok=True)
print(f"✅ Successfully configured. Model will be saved to: {repo_id}")


print("✅ Base model and tokenizer loaded.")

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})
    model.resize_token_embeddings(len(tokenizer))
    tokenizer.padding_side = "right"

print("\n--- Applying LoRA PEFT to the Base Model ---")
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth", random_state=3407,
)
print("✅ LoRA configured.")


# ===============================================================
# 2. ARCHITECTURE: The Corrected TitanReasoner Wrapper
# ===============================================================
from titans_pytorch.neural_memory import NeuralMemory
from torch.amp import custom_fwd
import math
import torch.nn.functional as F

class ManualLayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        gamma, beta = self.gamma, self.beta
        if gamma.ndim > 1:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return gamma * (x - mean) / (std + self.eps) + beta

class BatchedLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        weight, bias = self.weight, self.bias
        if weight.ndim > 2:
            x = torch.einsum('...ni,...oi->...no', x, weight)
        else:
            x = torch.einsum('...i,oi->...o', x, weight)
        if bias is not None:
            if bias.ndim == x.ndim - 1:
                bias = bias.unsqueeze(-2)
            x = x + bias
        return x

class EagerMemoryMLP(nn.Module):
    def __init__(self, dim, mult=4, depth=1):
        super().__init__()
        layers = []
        for _ in range(depth):
            layers.append(nn.Sequential(BatchedLinear(dim, dim * mult), nn.GELU(), BatchedLinear(dim * mult, dim)))
        self.model = nn.Sequential(*layers)
        self.norm = ManualLayerNorm(dim)

    def forward(self, x):
        return self.norm(self.model(x))

class PatchedNeuralMemory(NeuralMemory):
    def __init__(self, *args, **kwargs):
        dim = kwargs.get('dim')
        dim_head = kwargs.get('dim_head', dim)
        mlp_depth = kwargs.get('mem_mlp_depth', 1)
        eager_model = EagerMemoryMLP(dim=dim_head, depth=mlp_depth)
        kwargs['model'] = eager_model
        kwargs['mem_model_norm_add_residual'] = False
        kwargs['per_head_learned_parameters'] = False
        super().__init__(*args, **kwargs)
        self.store_norm = nn.LayerNorm(dim)
        self.retrieve_norm = nn.LayerNorm(dim)

class PatchedNeuralMemoryFP32(PatchedNeuralMemory):
    @custom_fwd(device_type='cuda', cast_inputs=torch.float32)
    def forward(self, *args, **kwargs):
        return super().forward(*args, **kwargs)

class TitanReasoner(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        model_dim = self.base_model.config.hidden_size
        print(f"--- Initializing Titans Neural Memory (dim={model_dim}) ---")
        self.memory = PatchedNeuralMemoryFP32(dim=model_dim, chunk_size=128, heads=4, dim_head=model_dim // 8)
        print("✅ Patched Neural Memory is online.")

    def forward(self, input_ids, attention_mask, labels=None, memory_state=None):
        input_embeds = self.base_model.get_input_embeddings()(input_ids)
        retrieved_memory, next_memory_state = self.memory(input_embeds, state=memory_state)
        augmented_embeds = input_embeds + retrieved_memory.to(input_embeds.dtype)
        outputs = self.base_model(inputs_embeds=augmented_embeds, attention_mask=attention_mask, labels=labels)
        return outputs, next_memory_state

# ===============================================================
# 3. DATA PREPARATION
# ===============================================================
print("\n--- Loading and Preparing Dataset ---")
# This assumes the dataset file from the previous step is present
dataset = load_dataset("json", data_files="purified_reasoner_dataset.jsonl", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

def formatting_and_tokenizing_collate_fn(batch):
    raw_texts = [tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False) for ex in batch]
    model_inputs = tokenizer(raw_texts, max_length=256, padding="max_length", truncation=True, return_tensors="pt")
    labels = model_inputs.input_ids.clone()
    prompt_lengths = []
    for ex in batch:
        prompt_messages = [msg for msg in ex["messages"] if msg['role'] != 'assistant']
        prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=False)
        if prompt_text.endswith(tokenizer.bos_token):
             prompt_text = prompt_text[:-len(tokenizer.bos_token)]
        prompt_tokens = tokenizer(prompt_text, return_tensors="pt").input_ids
        prompt_lengths.append(prompt_tokens.shape[1])
    for i in range(len(labels)):
        labels[i, :prompt_lengths[i]] = -100
    model_inputs["labels"] = labels
    return model_inputs

train_loader = DataLoader(train_dataset, batch_size=1, collate_fn=formatting_and_tokenizing_collate_fn)
eval_loader = DataLoader(eval_dataset, batch_size=1, collate_fn=formatting_and_tokenizing_collate_fn)
print("✅ DataLoaders are ready.")

# ===============================================================
# 4. TRAINING LOGIC
# ===============================================================
from torch.amp import autocast, GradScaler

print("\n--- Initializing Titan-Reasoner and Optimizer ---")
titan_reasoner = TitanReasoner(model).to("cuda")

# --- HYPERPARAMETER CHANGES ---
learning_rate = 5e-5
gradient_accumulation_steps = 8
num_epochs = 1

# --- CORRECTED SCHEDULER CALCULATION ---
# We calculate the total training steps based on the length of the dataloader for one full epoch
total_training_steps = len(train_loader) // gradient_accumulation_steps
warmup_steps = int(0.1 * total_training_steps) # Use 10% of total steps for warmup

optimizer = AdamW(titan_reasoner.parameters(), lr=learning_rate)
scaler = GradScaler('cuda')

# Now, provide the required arguments to the scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

# Initialize WandB with new config
wandb.init(project="titan-reasoner-fuzzer", name=f"run-{model_name}-v4-full-epoch", config={
    "learning_rate": learning_rate,
    "scheduler": "linear_with_warmup",
    "epochs": num_epochs,
    "batch_size": 1,
    "gradient_accumulation": gradient_accumulation_steps,
    "effective_batch_size": 1 * gradient_accumulation_steps
})


print("\n--- Starting Fine-Tuning with Neural Memory ---")
for epoch in range(num_epochs):
    titan_reasoner.train()
    total_loss = 0
    
    # CORRECTED: Removed `total=max_steps` so tqdm uses the length of train_loader
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    # REMOVED: No longer need to manually check for max_steps
    for step, batch in enumerate(progress_bar):
        # if step >= max_steps: break # <-- This line is no longer needed
        
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        labels = batch['labels'].to('cuda')
        
        with autocast(device_type='cuda'):
            outputs, _ = titan_reasoner(input_ids=input_ids, attention_mask=attention_mask, labels=labels, memory_state=None)
            loss = outputs.loss
            
        if loss is not None:
            scaled_loss = loss / gradient_accumulation_steps
            scaler.scale(scaled_loss).backward()
            
            if (step + 1) % gradient_accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
            wandb.log({"step_loss": loss.item()})
            
        if step % 20 == 0: torch.cuda.empty_cache()

    # The calculation for avg_train_loss needs a small adjustment as well
    avg_train_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} - Average Training Loss: {avg_train_loss:.4f}")

    titan_reasoner.eval()
    total_eval_loss = 0
    eval_steps = 0
    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to('cuda')
            attention_mask = batch['attention_mask'].to('cuda')
            labels = batch['labels'].to('cuda')
            with autocast(device_type='cuda'):
                outputs, _ = titan_reasoner(input_ids=input_ids, attention_mask=attention_mask, labels=labels, memory_state=None)
            if outputs.loss is not None:
                total_eval_loss += outputs.loss.item()
                eval_steps += 1
    avg_eval_loss = total_eval_loss / eval_steps if eval_steps > 0 else 0
    print(f"Epoch {epoch+1} - Validation Loss: {avg_eval_loss:.4f}")
    wandb.log({"epoch": epoch, "avg_train_loss": avg_train_loss, "avg_eval_loss": avg_eval_loss})

print("\n✅ Titan-Reasoner fine-tuning complete!")
wandb.finish()

# ===============================================================
# 5. SAVE AND UPLOAD MODEL
# ===============================================================
print("\n--- Saving model to Hugging Face Hub ---")

# Save LoRA adapters and tokenizer
titan_reasoner.base_model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

# Save the Titan memory module state
memory_path = "titan_reasoner_memory.pt"
torch.save(titan_reasoner.memory.state_dict(), memory_path)

# Upload the memory module to the same repository
upload_file(
    path_or_fileobj=memory_path,
    path_in_repo="titan_reasoner_memory.pt",
    repo_id=repo_id,
    token=HF_TOKEN,
)

print(f"✅ Model and memory module successfully saved to: {repo_id}")

✅ Successfully configured. Model will be saved to: surfiniaburger/Purified-Reasoner-gpt-oss-20b-v2
✅ Base model and tokenizer loaded.

--- Applying LoRA PEFT to the Base Model ---
Unsloth: Making `model.base_model.model.model` require gradients
✅ LoRA configured.

--- Loading and Preparing Dataset ---


Generating train split: 0 examples [00:00, ? examples/s]

✅ DataLoaders are ready.

--- Initializing Titan-Reasoner and Optimizer ---
--- Initializing Titans Neural Memory (dim=2880) ---
✅ Patched Neural Memory is online.


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Weave is installed but not imported. Add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/



--- Starting Fine-Tuning with Neural Memory ---


Epoch 1/1:   0%|          | 0/450 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARAL